# 🇧🇷 Fine-tuning de Modelo para Geração de Texto em Português

## O que você vai aprender neste notebook?

Este notebook te ensina como **treinar um modelo para gerar texto** no estilo de um livro específico.

### Diferença entre este notebook e o anterior:

**Notebook anterior (bertimbau.ipynb):**
- Tarefa: **Classificação** (decidir se um texto é bíblico ou não)
- Modelo: BERT (entende texto)
- Saída: Probabilidades (0 ou 1)

**Este notebook (treino_atos.ipynb):**
- Tarefa: **Geração** (criar novo texto no estilo do livro)
- Modelo: GPT-2 ou similar (gera texto)
- Saída: Texto novo gerado pelo modelo

### O que vamos fazer?
1. **Carregar um livro** (ex: Atos dos Apóstolos)
2. **Preparar os dados** - Dividir em parágrafos
3. **Configurar o modelo** - GPT-2 com LoRA
4. **Treinar** - Deixar o modelo aprender o estilo do livro
5. **Gerar texto** - Criar novo texto no mesmo estilo

### Técnicas que usaremos:
- **QLoRA**: LoRA + Quantização (ainda mais eficiente!)
- **Causal Language Modeling**: Prever a próxima palavra
- **SFTTrainer**: Trainer otimizado para fine-tuning supervisionado

Vamos começar!

In [49]:
# ============================================================================
# PASSO 1: Importar Bibliotecas e Verificar GPU
# ============================================================================
# Aqui importamos todas as ferramentas necessárias.
# Também verificamos se temos GPU disponível (muito mais rápido!).

import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,           # Modelo para gerar texto
    AutoTokenizer,                  # Converte texto em números
    BitsAndBytesConfig,             # Configuração de quantização
    TrainingArguments,              # Argumentos de treinamento
    Trainer,                        # Gerenciador de treinamento
    DataCollatorForLanguageModeling # Prepara dados para geração
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig  # Trainer otimizado para fine-tuning

# Verificar GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Dispositivo: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   ⚠️  GPU não detectada. Treinamento será MUITO mais lento.")

# Configurações
BOOK_PATH = './biblia.txt'          # Arquivo com o texto para treinar
MODEL_NAME = "gpt2"                 # Modelo a usar (GPT-2 é leve e rápido)
OUTPUT_DIR = "./results_mistral_book"  # Onde salvar o modelo

print(f"\n📚 Configurações:")
print(f"   Arquivo: {BOOK_PATH}")
print(f"   Modelo: {MODEL_NAME}")
print(f"   Saída: {OUTPUT_DIR}")

Dispositivo: cuda
Nome da GPU: NVIDIA GeForce RTX 3060 Ti


📖 Etapa 2: Carregar e Preparar o Livro
O texto do livro será carregado e transformado em um formato de dataset.

In [50]:
# ============================================================================
# PASSO 2: Carregar e Preparar o Livro
# ============================================================================
# Vamos ler o arquivo de texto e transformá-lo em um dataset.
#
# O que vamos fazer:
# 1. Ler o arquivo
# 2. Dividir em parágrafos (chunks)
# 3. Filtrar parágrafos muito curtos
# 4. Criar um dataset estruturado
# 5. Dividir em treino e validação

print("📖 Carregando livro...")

try:
    with open(BOOK_PATH, 'r', encoding='utf-8') as f:
        book_text = f.read()
    print(f"✅ Livro carregado! Tamanho: {len(book_text):,} caracteres")
except FileNotFoundError:
    print(f"❌ Arquivo '{BOOK_PATH}' não encontrado!")
    print("   Usando texto de exemplo para demonstração...")
    book_text = "Era uma vez uma cidade distante, onde as palavras tinham o peso de ouro. "
    book_text += "O protagonista, um jovem chamado Elias, descobriu um segredo ancestral. "
    book_text += "Ele começou a registrar suas descobertas em um diário escondido."

# Dividir em parágrafos (separados por linhas vazias)
print("\n✂️  Dividindo em parágrafos...")
text_chunks = [
    chunk.strip()
    for chunk in book_text.split('\n\n')
    if len(chunk.strip()) > 50  # Filtrar parágrafos muito curtos
]

print(f"   Total de parágrafos: {len(text_chunks)}")
print(f"   Exemplo de parágrafo:")
print(f"   '{text_chunks[0][:100]}...'")

# Criar dataset
print("\n📊 Criando dataset...")
data = {'text': text_chunks}
dataset = Dataset.from_dict(data)

# Dividir em treino (99%) e validação (1%)
train_test_split = dataset.train_test_split(test_size=0.01, seed=42)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

print(f"\n✅ Dataset criado!")
print(f"   Total: {len(dataset)} parágrafos")
print(f"   Treino: {len(train_dataset)} parágrafos")
print(f"   Validação: {len(eval_dataset)} parágrafos")

Total de amostras (parágrafos/chunks): 405
Tamanho do treino: 400
Tamanho da avaliação: 5


⚙️ Etapa 3: Configuração do Modelo e Tokenização (QLoRA)
Esta é a etapa mais crítica. Usaremos o 4-bit quantization (BitsAndBytes) e o LoRA para reduzir drasticamente o uso de memória da GPU, permitindo o fine-tuning do modelo Mistral 7B.

In [51]:
# ============================================================================
# PASSO 3: Configurar Modelo com QLoRA (Quantização + LoRA)
# ============================================================================
# QLoRA é uma combinação de duas técnicas para máxima eficiência:
#
# 1. **Quantização (4-bit)**:
#    - Reduz o tamanho do modelo em 4x
#    - Usa menos memória GPU
#    - Quase nenhuma perda de qualidade
#
# 2. **LoRA**:
#    - Treina apenas adaptadores pequenos
#    - Mantém o modelo original congelado
#
# Resultado: Podemos treinar modelos GRANDES em GPUs pequenas!

print("⚙️  Configurando quantização 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                  # Carregar em 4-bit
    bnb_4bit_quant_type="nf4",         # Tipo de quantização (Normal Float 4-bit)
    bnb_4bit_compute_dtype=torch.bfloat16,  # Tipo de computação
    bnb_4bit_use_double_quant=True,     # Quantização dupla (mais eficiente)
)

print("🔤 Carregando tokenizador...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # Usar EOS como padding

print("🤖 Carregando modelo com quantização...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,     # Aplicar quantização
    device_map="auto"                  # Mapear automaticamente para GPU
)

print("\n⚙️  Configurando LoRA...")
lora_config = LoraConfig(
    lora_alpha=16,                      # Escala dos pesos LoRA
    lora_dropout=0.1,                   # Dropout para regularização
    r=64,                               # Rank LoRA (dimensão dos adaptadores)
    bias="none",                       # Não treinar bias
    task_type="CAUSAL_LM",             # Tipo de tarefa: Geração de Texto
)

# Preparar modelo para k-bit training
print("\n🔧 Preparando modelo para treinamento...")
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

print("\n✅ Modelo configurado com QLoRA!")
print("\n📊 Estatísticas:")
model.print_trainable_parameters()
print("\n💡 Explicação:")
print("   - trainable params: Parâmetros que serão treinados (LoRA)")
print("   - all params: Total de parâmetros (quantizado em 4-bit)")
print("   - trainable%: Percentual a treinar (deve ser ~1-2%)")


--- Modelo configurado com QLoRA ---
trainable params: 2,359,296 || all params: 126,799,104 || trainable%: 1.8607


# ============================================================================
# PASSO 4: Configurar e Executar o Treinamento
# ============================================================================
# Agora vamos treinar o modelo para gerar texto no estilo do livro.
#
# O que acontece:
# 1. O modelo recebe um parágrafo
# 2. Aprende a prever a próxima palavra
# 3. Repete para cada palavra do parágrafo
# 4. Ajusta os pesos LoRA para melhorar
# 5. Repete com o próximo parágrafo
#
# Depois de cada época, testamos com dados de validação.
#
# O SFTTrainer (Supervised Fine-Tuning Trainer) é otimizado para este tipo de tarefa.

In [53]:
# ====================================================================
# 🔹 Etapa 4: Argumentos de Treinamento e Inicialização do Trainer
# ====================================================================

# 1. Argumentos de Treinamento
# ATENÇÃO: Estes argumentos são leves. Ajuste conforme o tamanho do seu livro e GPU!
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,   # Ajuste para caber na sua GPU (4 é conservador)
    gradient_accumulation_steps=4,   # Aumenta o batch size efetivo para 16 (4*4)
    optim="paged_adamw_32bit",       # Otimizador otimizado para QLoRA
    logging_steps=10,
    learning_rate=3e-4,              # Taxa de aprendizado baixa para fine-tuning
    fp16=False,                      # Use True se sua GPU suportar, mas bfloat16 já foi setado
    bf16=True,                       # Usar bfloat16 para precisão e velocidade
    max_steps=500,                   # Número máximo de passos (ajuste para o tamanho do livro)
    warmup_ratio=0.03,
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    dataloader_drop_last=True,
    
)

sftconfig = SFTConfig(max_length=1024,
    packing=True,
    dataset_text_field="text")

# 2. Inicializar SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    args=training_args   # Argumentos de Otimização/Execução
    )

# 3. Iniciar Treinamento
print("\n--- Iniciando Treinamento ---")
trainer.train()

# 4. Salvar o Modelo Ajustado (apenas os pesos LoRA)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n✅ Fine-Tuning concluído! Modelo e Tokenizador salvos em {OUTPUT_DIR}")

/opt/conda/envs/bert/lib/python3.10/site-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/opt/conda/envs/bert/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Truncating eval dataset: 100%|██████████| 5/5 [00:00<00:00, 4410.41 examples/s]



--- Iniciando Treinamento ---


/opt/conda/envs/bert/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,4.836300,No log
200,4.505200,No log
300,4.339400,No log
400,4.305000,No log
500,4.246800,No log


/opt/conda/envs/bert/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/envs/bert/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/envs/bert/lib/python3.1


✅ Fine-Tuning concluído! Modelo e Tokenizador salvos em ./results_mistral_book


# ============================================================================
# PASSO 5: Gerar Texto com o Modelo Treinado
# ============================================================================
# Agora vamos usar o modelo para GERAR novo texto no estilo do livro!
#
# O que vamos fazer:
# 1. Carregar o modelo base + pesos LoRA
# 2. Fundir os pesos (merge) para ter um modelo único
# 3. Dar um "prompt" (início de frase)
# 4. Deixar o modelo continuar a frase
# 5. Ver o texto gerado
#
# Exemplo:
#   Prompt: "Saudae a Amplias"
#   Geração: "Saudae a Amplias todos que présem, ou não..."
#
# O modelo aprendeu o estilo e vocabulário do livro!

In [56]:
# ====================================================================
# 🔹 Etapa 5: Inferência e Geração de Texto
# ====================================================================
from peft import PeftModel

# 1. Carregar o modelo base original e os pesos LoRA treinados
# Carregamos o modelo base novamente na CPU, sem quantização (para o PeftModel)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)

# 2. Adicionar os pesos LoRA ao modelo base
model_to_infer = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model_to_infer = model_to_infer.merge_and_unload() # Junta os pesos LoRA no modelo base
model_to_infer.to(device).eval() # Manda para GPU e modo de avaliação

print("\n--- Modelo pronto para Geração de Texto ---")

def generate_book_text(prompt, max_new_tokens=100):
    # Tokeniza o prompt
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Gera a continuação do texto
    with torch.no_grad():
        outputs = model_to_infer.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,          # Amostragem para geração criativa
            top_k=50,
            top_p=1,
            temperature=0.7,         # Nível de criatividade
            pad_token_id=tokenizer.eos_token_id
        )
        
    # Decodifica a saída
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

# 3. Testes de Geração
prompt1 = "Saudae a Amplias"
print(f"\nPrompt: {prompt1}")
print("Geração:\n" + generate_book_text(prompt1))

prompt2 = "Saudam-vos"
print(f"\nPrompt: {prompt2}")
print("Geração:\n" + generate_book_text(prompt2))


--- Modelo pronto para Geração de Texto ---

Prompt: Saudae a Amplias
Geração:
Saudae a Amplias todos que présem, ou não
daniel, eu quesam eu pela fas, eu vivos eu do
daniel, eu vivos  do vivos eu pela fas, eu vivos eu pela fas, eu vivos eu pela fas, eu vivos eu pela fas, eu vivos eu p

Prompt: Saudam-vos
Geração:
Saudam-vos vos,  por que vos  aos de Deus, que que aos, que vos
divor, tambem que vos de Deus.
